# Laboratorium 6: prywatność, przejrzystość i uczciwość w systemach opartych na wiedzy

Notebook do instrukcji `lab06_instrukcja.md`. Dane osobowe w grafie uczelni i w populacji studentów są fikcyjne. Zbiór Adult pochodzi z OpenML (spis powszechny USA z 1994 r., dane zanonimizowane u źródła).

In [ ]:
%pip install -q rdflib owlrl pgmpy

In [ ]:
import time, itertools, hashlib, math
from functools import reduce
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rdflib, owlrl, pgmpy, sklearn
from rdflib import Graph, Namespace, Literal, URIRef, RDF, RDFS, OWL, XSD

RANDOM_STATE = 42
U = Namespace("http://example.org/uczelnia#")
pd.set_option("display.width", 140)
print("rdflib", rdflib.__version__, "| owlrl", owlrl.__version__, "| pgmpy", pgmpy.__version__, "| scikit-learn", sklearn.__version__)

## Zadanie 1: wnioskowanie jako zagrożenie dla prywatności

Graf uczelni ze wspólnego słownictwa kursu, rozszerzony o quasi-identyfikatory (płeć, rok urodzenia, kod pocztowy, kierunek) i dwa rodzaje danych wrażliwych: orzeczenie o niepełnosprawności (klasa `u:OsobaZOrzeczeniem`) oraz stypendium socjalne. TBox i ABox są w osobnych napisach, bo w zadaniu 3 schemat będzie publiczny, a dane nie.

In [ ]:
PREFIKSY = """
@prefix u:    <http://example.org/uczelnia#> .
@prefix rdf:  <http://www.w3.org/1999/02/22-rdf-syntax-ns#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix owl:  <http://www.w3.org/2002/07/owl#> .
@prefix xsd:  <http://www.w3.org/2001/XMLSchema#> .
"""

TBOX_TTL = """
u:Osoba a owl:Class .
u:Student a owl:Class ; rdfs:subClassOf u:Osoba .
u:Pracownik a owl:Class ; rdfs:subClassOf u:Osoba .
u:Kurs a owl:Class .
u:KursObowiazkowy a owl:Class ; rdfs:subClassOf u:Kurs .
u:KursObieralny a owl:Class ; rdfs:subClassOf u:Kurs .
u:Wydzial a owl:Class .
u:PracaDyplomowa a owl:Class .
u:Wykladowca a owl:Class ;
    owl:equivalentClass [ a owl:Class ;
        owl:intersectionOf ( u:Pracownik
            [ a owl:Restriction ; owl:onProperty u:prowadzi ; owl:someValuesFrom u:Kurs ] ) ] .

u:prowadzi a owl:ObjectProperty ; rdfs:domain u:Pracownik ; rdfs:range u:Kurs .
u:zapisanyNa a owl:ObjectProperty ; rdfs:domain u:Student ; rdfs:range u:Kurs .
u:zaliczyl a owl:ObjectProperty ; rdfs:domain u:Student ; rdfs:range u:Kurs .
u:wymagaKursu a owl:ObjectProperty , owl:TransitiveProperty ; rdfs:domain u:Kurs ; rdfs:range u:Kurs .
u:naWydziale a owl:ObjectProperty ; rdfs:range u:Wydzial .
u:promotor a owl:ObjectProperty ; rdfs:domain u:PracaDyplomowa ; rdfs:range u:Pracownik .
u:autor a owl:ObjectProperty ; rdfs:domain u:PracaDyplomowa ; rdfs:range u:Student .
u:imie a owl:DatatypeProperty . u:nazwisko a owl:DatatypeProperty .
u:rokStudiow a owl:DatatypeProperty . u:ects a owl:DatatypeProperty .
u:rokUrodzenia a owl:DatatypeProperty .

# rozszerzenie lab 6: quasi-identyfikatory i dane wrażliwe
u:plec a owl:DatatypeProperty . u:kodPocztowy a owl:DatatypeProperty .
u:kierunek a owl:DatatypeProperty .
u:stypendiumSocjalne a owl:DatatypeProperty ; rdfs:range xsd:boolean .
u:OsobaZOrzeczeniem a owl:Class ; rdfs:subClassOf u:Student ;
    rdfs:comment "Student z orzeczeniem o niepełnosprawności (dane o zdrowiu)" .
u:KursAdaptacyjny a owl:Class ; rdfs:subClassOf u:KursObieralny ;
    rdfs:comment "Kurs dostępny wyłącznie dla osób z orzeczeniem" .
[ a owl:Restriction ; owl:onProperty u:zapisanyNa ; owl:someValuesFrom u:KursAdaptacyjny ]
    rdfs:subClassOf u:OsobaZOrzeczeniem .
u:maAsystenta a owl:ObjectProperty ; rdfs:domain u:OsobaZOrzeczeniem ; rdfs:range u:Pracownik ;
    rdfs:comment "Asystent przydzielony przez biuro ds. osób z niepełnosprawnościami" .
"""

ABOX_TTL = """
u:WI a u:Wydzial ; rdfs:label "Wydział Informatyki" .
u:WZ a u:Wydzial ; rdfs:label "Wydział Zarządzania" .

u:Programowanie1 a u:KursObowiazkowy ; u:ects 6 ; u:naWydziale u:WI .
u:Programowanie2 a u:KursObowiazkowy ; u:ects 6 ; u:wymagaKursu u:Programowanie1 ; u:naWydziale u:WI .
u:Algorytmy a u:KursObowiazkowy ; u:ects 5 ; u:wymagaKursu u:Programowanie2 ; u:naWydziale u:WI .
u:SztucznaInteligencja a u:KursObieralny ; u:ects 5 ; u:wymagaKursu u:Algorytmy ; u:naWydziale u:WI .
u:Statystyka a u:KursObowiazkowy ; u:ects 4 ; u:naWydziale u:WZ .
u:Marketing a u:KursObieralny ; u:ects 3 ; u:naWydziale u:WZ .
u:WFAdaptacyjny a u:KursAdaptacyjny ; u:ects 2 ; u:naWydziale u:WZ .

u:p1 a u:Pracownik ; u:imie "Halina" ; u:nazwisko "Kowalczyk" ; u:prowadzi u:Programowanie1 , u:Programowanie2 .
u:p2 a u:Pracownik ; u:imie "Marek" ; u:nazwisko "Lis" ; u:prowadzi u:Algorytmy , u:SztucznaInteligencja .
u:p3 a u:Pracownik ; u:imie "Ewa" ; u:nazwisko "Sowa" ; u:prowadzi u:Statystyka , u:Marketing .
u:p4 a u:Pracownik ; u:imie "Tomasz" ; u:nazwisko "Baran" ; u:prowadzi u:WFAdaptacyjny .
u:p5 a u:Pracownik ; u:imie "Joanna" ; u:nazwisko "Wrona" .

u:s01 a u:Student ; u:imie "Anna" ; u:nazwisko "Nowak" ; u:plec "K" ; u:rokUrodzenia 2001 ; u:kodPocztowy "61-001" ; u:kierunek "informatyka" ; u:rokStudiow 2 ;
    u:zaliczyl u:Programowanie1 , u:Programowanie2 ; u:zapisanyNa u:Algorytmy .
u:s02 a u:Student ; u:imie "Piotr" ; u:nazwisko "Wiśniewski" ; u:plec "M" ; u:rokUrodzenia 2000 ; u:kodPocztowy "61-001" ; u:kierunek "informatyka" ; u:rokStudiow 2 ;
    u:zaliczyl u:Programowanie1 ; u:zapisanyNa u:Programowanie2 .
u:s03 a u:Student , u:OsobaZOrzeczeniem ; u:imie "Katarzyna" ; u:nazwisko "Wójcik" ; u:plec "K" ; u:rokUrodzenia 2002 ; u:kodPocztowy "61-245" ; u:kierunek "zarządzanie" ; u:rokStudiow 1 ;
    u:zapisanyNa u:Statystyka , u:WFAdaptacyjny ; u:stypendiumSocjalne true .
u:s04 a u:Student ; u:imie "Michał" ; u:nazwisko "Kamiński" ; u:plec "M" ; u:rokUrodzenia 2001 ; u:kodPocztowy "60-101" ; u:kierunek "informatyka" ; u:rokStudiow 3 ;
    u:zaliczyl u:Programowanie1 , u:Programowanie2 , u:Algorytmy ; u:zapisanyNa u:SztucznaInteligencja .
u:s05 a u:Student , u:OsobaZOrzeczeniem ; u:imie "Agnieszka" ; u:nazwisko "Lewandowska" ; u:plec "K" ; u:rokUrodzenia 2001 ; u:kodPocztowy "60-101" ; u:kierunek "informatyka" ; u:rokStudiow 3 ;
    u:zaliczyl u:Programowanie1 , u:Programowanie2 , u:Algorytmy ; u:zapisanyNa u:SztucznaInteligencja ; u:maAsystenta u:p5 .
u:s06 a u:Student ; u:imie "Jakub" ; u:nazwisko "Zieliński" ; u:plec "M" ; u:rokUrodzenia 2003 ; u:kodPocztowy "61-245" ; u:kierunek "zarządzanie" ; u:rokStudiow 1 ;
    u:zapisanyNa u:Statystyka , u:Marketing ; u:stypendiumSocjalne true .
u:s07 a u:Student ; u:imie "Magdalena" ; u:nazwisko "Szymańska" ; u:plec "K" ; u:rokUrodzenia 2002 ; u:kodPocztowy "62-020" ; u:kierunek "zarządzanie" ; u:rokStudiow 2 ;
    u:zaliczyl u:Statystyka ; u:zapisanyNa u:Marketing .
u:s08 a u:Student ; u:imie "Paweł" ; u:nazwisko "Woźniak" ; u:plec "M" ; u:rokUrodzenia 2000 ; u:kodPocztowy "61-001" ; u:kierunek "informatyka" ; u:rokStudiow 3 ;
    u:zaliczyl u:Programowanie1 , u:Programowanie2 ; u:zapisanyNa u:Algorytmy .
u:s09 a u:Student , u:OsobaZOrzeczeniem ; u:imie "Natalia" ; u:nazwisko "Dąbrowska" ; u:plec "K" ; u:rokUrodzenia 2003 ; u:kodPocztowy "62-020" ; u:kierunek "informatyka" ; u:rokStudiow 1 ;
    u:zapisanyNa u:Programowanie1 .
u:s10 a u:Student , u:OsobaZOrzeczeniem ; u:imie "Krzysztof" ; u:nazwisko "Kozłowski" ; u:plec "M" ; u:rokUrodzenia 2002 ; u:kodPocztowy "60-101" ; u:kierunek "zarządzanie" ; u:rokStudiow 2 ;
    u:zaliczyl u:Statystyka ; u:zapisanyNa u:Marketing , u:WFAdaptacyjny ; u:stypendiumSocjalne true .
u:s11 a u:Student ; u:imie "Aleksandra" ; u:nazwisko "Jankowska" ; u:plec "K" ; u:rokUrodzenia 2001 ; u:kodPocztowy "61-001" ; u:kierunek "informatyka" ; u:rokStudiow 2 ;
    u:zaliczyl u:Programowanie1 , u:Programowanie2 ; u:zapisanyNa u:Algorytmy .
u:s12 a u:Student ; u:imie "Tomasz" ; u:nazwisko "Mazur" ; u:plec "M" ; u:rokUrodzenia 2003 ; u:kodPocztowy "61-245" ; u:kierunek "informatyka" ; u:rokStudiow 1 ;
    u:zapisanyNa u:Programowanie1 .
u:s13 a u:Student ; u:imie "Monika" ; u:nazwisko "Krawczyk" ; u:plec "K" ; u:rokUrodzenia 2002 ; u:kodPocztowy "61-245" ; u:kierunek "zarządzanie" ; u:rokStudiow 1 ;
    u:zapisanyNa u:Statystyka .
u:s14 a u:Student ; u:imie "Łukasz" ; u:nazwisko "Piotrowski" ; u:plec "M" ; u:rokUrodzenia 2001 ; u:kodPocztowy "62-020" ; u:kierunek "zarządzanie" ; u:rokStudiow 3 ;
    u:zaliczyl u:Statystyka , u:Marketing .
u:s15 a u:Student ; u:imie "Zofia" ; u:nazwisko "Grabowska" ; u:plec "K" ; u:rokUrodzenia 2000 ; u:kodPocztowy "60-101" ; u:kierunek "zarządzanie" ; u:rokStudiow 3 ;
    u:zaliczyl u:Statystyka , u:Marketing .
u:s16 a u:Student ; u:imie "Adam" ; u:nazwisko "Pawlak" ; u:plec "M" ; u:rokUrodzenia 2002 ; u:kodPocztowy "61-001" ; u:kierunek "informatyka" ; u:rokStudiow 2 ;
    u:zaliczyl u:Programowanie1 ; u:zapisanyNa u:Programowanie2 .

u:pd1 a u:PracaDyplomowa ; u:autor u:s04 ; u:promotor u:p2 .
u:pd2 a u:PracaDyplomowa ; u:autor u:s15 ; u:promotor u:p3 .
"""

TBOX = Graph().parse(data=PREFIKSY + TBOX_TTL, format="turtle")
g = Graph().parse(data=PREFIKSY + TBOX_TTL + ABOX_TTL, format="turtle")
print("TBox:", len(TBOX), "trójek, cały graf:", len(g), "trójek")

In [ ]:
# Napiszcie funkcję domknij(graf), która zwraca kopię grafu z domknięciem OWL 2 RL (owlrl.DeductiveClosure).
# Sprawdźcie liczbę trójek przed i po oraz czas działania.

### 1a. Publikacja „zanonimizowanego” grafu

Dziekanat chce udostępnić graf do badań nad obciążeniem kursów. Usuwa imiona, nazwiska, stypendium i jawny typ `OsobaZOrzeczeniem`, a identyfikatory studentów zastępuje pseudonimami.

In [ ]:
# Zbudujcie graf pub: pseudonimy zamiast IRI studentów (np. skrót SHA-256 z solą),
# bez imion, nazwisk, stypendium i bez jawnego typu u:OsobaZOrzeczeniem.
# Zapytajcie SPARQL o ?x a u:OsobaZOrzeczeniem przed i po domknięciu. Porównajcie z faktycznymi danymi.

In [ ]:
# Dla każdego wywnioskowanego pseudonimu wypiszcie trójki, które posłużyły za przesłanki.

### 1b. Łączenie z zewnętrznym zbiorem

Samorząd opublikował wyniki uczelnianego biegu: imię, nazwisko, płeć, rok urodzenia i kod pocztowy (lista poniżej jest gotowa). Połączcie ją z publikacją po wspólnych atrybutach.

In [ ]:
bieg = pd.DataFrame([
    ("Katarzyna", "Wójcik", "K", 2002, "61-245"),
    ("Agnieszka", "Lewandowska", "K", 2001, "60-101"),
    ("Krzysztof", "Kozłowski", "M", 2002, "60-101"),
    ("Monika", "Krawczyk", "K", 2002, "61-245"),
    ("Anna", "Nowak", "K", 2001, "61-001"),
    ("Paweł", "Woźniak", "M", 2000, "61-001"),
    ("Robert", "Kaczmarek", "M", 1999, "61-001"),
    ("Ewelina", "Zając", "K", 2000, "62-020"),
], columns=["imie", "nazwisko", "plec", "rokUrodzenia", "kodPocztowy"])
bieg

In [ ]:
# Wyciągnijcie z pub zapytaniem SPARQL tabelę: pseudonim, plec, rokUrodzenia, kodPocztowy, kierunek.
# Dołączcie kolumnę z informacją, czy reasoner wywnioskował orzeczenie.
# Połączcie z tabelą bieg (merge po plec, rokUrodzenia, kodPocztowy). Ilu kandydatów ma każda osoba z listy?

In [ ]:
# Dla każdego niepustego podzbioru quasi-identyfikatorów policzcie, ilu studentów jest w nim unikalnych
# i jaki jest rozmiar najmniejszej klasy równoważności (k).

### Wnioski do zadania 1

(uzupełnij)

## Zadanie 2: k-anonimowość i l-różnorodność na zbiorze Adult

Quasi-identyfikatory: `age`, `sex`, `race`, `marital-status`, `native-country`. Atrybut wrażliwy: `income` (dochód powyżej 50 tys. USD). Hierarchie generalizacji są gotowe, każda zaczyna się od poziomu 0 (wartość oryginalna) i kończy na `*`.

In [ ]:
from sklearn.datasets import fetch_openml
# Wczytajcie fetch_openml("adult", version=2, as_frame=True), usuńcie braki (dropna),
# zmieńcie nazwę kolumny "class" na "income", kolumny kategoryczne zamieńcie na str.

In [ ]:
def g_age(v, poziom):
    if poziom == 0:
        return v
    if poziom == 4:
        return "*"
    w = {1: 5, 2: 10, 3: 20}[poziom]
    lo = (v // w) * w
    return f"{lo}-{lo + w - 1}"

MALZ = {"Married-civ-spouse": "Married", "Married-AF-spouse": "Married", "Married-spouse-absent": "Married",
        "Never-married": "Never-married", "Divorced": "Prev-married", "Separated": "Prev-married", "Widowed": "Prev-married"}
gwiazdka = lambda s: pd.Series("*", index=s.index)
HIER = {
    "age": [lambda s: s] + [lambda s, p=p: s.map(lambda v: g_age(v, p)) for p in range(1, 5)],
    "sex": [lambda s: s, gwiazdka],
    "race": [lambda s: s, lambda s: s.where(s == "White", "Other"), gwiazdka],
    "marital-status": [lambda s: s, lambda s: s.map(MALZ), gwiazdka],
    "native-country": [lambda s: s, lambda s: s.where(s == "United-States", "Other"), gwiazdka],
}
QI = list(HIER)
MAXP = {q: len(HIER[q]) - 1 for q in QI}

def generalizuj(d, wezel):
    out = d.copy()
    for q, p in zip(QI, wezel):
        out[q] = HIER[q][p](d[q])
    return out

print({q: MAXP[q] for q in QI})

In [ ]:
# Podzielcie dane 70/30 ze stratyfikacją po income (random_state=RANDOM_STATE).
# Policzcie k dla danych oryginalnych i liczbę wierszy unikalnych względem QI.

In [ ]:
# Przejdźcie po całej kracie generalizacji (iloczyn poziomów wszystkich QI).
# Dla każdego węzła zapamiętajcie, dla każdego wiersza, rozmiar jego klasy równoważności
# i liczbę różnych wartości income w tej klasie.
# Napiszcie funkcję strata(węzeł, liczba_usuniętych) i funkcję najlepszy(k, l, max_supr=0.01),
# która zwraca węzeł o najmniejszej stracie spełniający k-anonimowość (i l-różnorodność) po supresji do 1% wierszy.

In [ ]:
# Dla k w [2, 5, 10, 25, 50, 100] znajdźcie najlepszy węzeł i zbudujcie tabelę:
# k, węzeł, strata, liczba usuniętych wierszy, liczba wierszy w klasach jednorodnych względem income,
# zbalansowana trafność drzewa (max_depth=8) uczonego na danych zanonimizowanych:
#   (a) cechy QI + education-num, hours-per-week, workclass, occupation, relationship,
#   (b) tylko QI.
# Zbiór testowy generalizujcie tym samym węzłem. Porównajcie z modelem na danych oryginalnych.

In [ ]:
# Atak homogeniczności dla k = 5: wypiszcie największe klasy równoważności, w których wszyscy mają to samo income.
# Pokażcie konkretną osobę: co wie atakujący (QI) i czego się dowiaduje.

In [ ]:
# Powtórzcie wyszukiwanie z warunkiem 2-różnorodności (co najmniej 2 różne wartości income w klasie)
# dla k w [2, 5, 10, 25]. Jak zmienia się wybrany węzeł i strata?

In [ ]:
# Narysujcie stratę i udział wierszy w klasach jednorodnych w funkcji k (oś x logarytmiczna).

### Wnioski do zadania 2

(uzupełnij)

## Zadanie 3: kontrola dostępu do grafu wiedzy

Polityka dostępu:

| rola | widzi |
|---|---|
| student | katalog kursów z prowadzącymi, listy zapisanych na kursy (imię, nazwisko), wszystkie własne dane |
| prowadzący | katalog, studentów zapisanych na swoje kursy (z informacją o asystencie), swoje prace dyplomowe |
| dziekanat | cały graf |

Fakty wrażliwe: przynależność do `u:OsobaZOrzeczeniem` i `u:stypendiumSocjalne`. Poza dziekanatem zna je tylko sama osoba, której dotyczą. TBox jest publiczny.

In [ ]:
# Zdefiniujcie zapytania CONSTRUCT realizujące politykę dla ról student i prowadzący
# (parametr ?ja przekazywany przez initBindings). Dziekanat widzi cały graf.
# Napiszcie funkcję wycieki(widok, ja), która zwraca fakty wrażliwe o innych osobach:
#   (a) jawnie obecne w widoku, (b) obecne po domknięciu widoku razem z publicznym TBox.
# Zbudujcie tabelę dla użytkowników s01, s03, s12, p1, p2, p3, p4.

In [ ]:
# Napiszcie funkcję napraw(widok, ja), która usuwa z widoku jak najmniej trójek ABox tak,
# żeby domknięcie (widok + TBox) nie zawierało faktów wrażliwych o innych osobach.
# Wskazówka: usuwajcie zachłannie trójki o osobach, których dotyczy wyciek, a potem spróbujcie przywrócić każdą z usuniętych.
# Wypiszcie dla każdego użytkownika liczbę wycieków przed i po naprawie oraz usunięte trójki.

### Wnioski do zadania 3

(uzupełnij)

## Zadanie 4: objaśnianie decyzji

Populacja 3000 fikcyjnych studentów i regulamin stypendium za wyniki w postaci reguł. Ten sam zbiór danych posłuży w zadaniu 5. Zmienna `wyroznienie` oznacza, że student faktycznie należy do najlepszych (to, co regulamin próbuje uchwycić); system jej nie widzi.

In [ ]:
def populacja(n=3000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    plec = rng.choice(["K", "M"], size=n)
    kierunki = ["informatyka", "matematyka", "psychologia", "zarządzanie"]
    p_k = {"K": [0.15, 0.20, 0.35, 0.30], "M": [0.45, 0.20, 0.10, 0.25]}
    kierunek = np.array([rng.choice(kierunki, p=p_k[p]) for p in plec])
    zdolnosci = rng.normal(0, 1, n)
    urlop = rng.random(n) < np.where(plec == "K", 0.20, 0.05)
    srednia = np.clip(3.95 + 0.33 * zdolnosci + rng.normal(0, 0.18, n) - 0.10 * urlop, 3.0, 5.0).round(2)
    ects = rng.random(n) < 1 / (1 + np.exp(-(1.6 + zdolnosci - urlop)))
    p_h = pd.Series(kierunek).map(dict(zip(kierunki, [0.50, 0.25, 0.04, 0.06]))).to_numpy()
    hackathon = rng.random(n) < (p_h * 2 / (1 + np.exp(-zdolnosci))).clip(0, 1)
    kolo = rng.random(n) < 1 / (1 + np.exp(-(-0.9 + 0.9 * zdolnosci)))
    wyroznienie = (zdolnosci + rng.normal(0, 0.5, n)) > 1.0
    return pd.DataFrame({"plec": plec, "kierunek": kierunek, "srednia": srednia, "ects_w_terminie": ects,
                         "urlop": urlop, "hackathon": hackathon, "kolo": kolo, "wyroznienie": wyroznienie})

# regulamin: reguły sprawdzane po kolei, pierwsza spełniona rozstrzyga
REGULY = [
    ("R1 urlop dziekański w roku oceny", [("urlop", "==", True)], False),
    ("R2 wysoka średnia", [("srednia", ">=", 4.6), ("ects_w_terminie", "==", True)], True),
    ("R3 dobra średnia i hackathon", [("srednia", ">=", 4.2), ("ects_w_terminie", "==", True), ("hackathon", "==", True)], True),
]
DOMYSLNIE = False

pop = populacja()
print(pop.shape)
pop.head()

In [ ]:
# Napiszcie silnik wnioskuj(r, reguly), który przechodzi reguły po kolei i zwraca decyzję oraz ślad:
# dla każdej sprawdzonej reguły listę warunków z wartością faktyczną i wynikiem sprawdzenia.
# Napiszcie funkcję dlaczego(r), która drukuje ślad, a przy odmowie odpowiada na pytanie "dlaczego nie":
# dla każdej reguły przyznającej wypisuje niespełnione warunki albo regułę, która rozstrzygnęła wcześniej.

In [ ]:
# Napiszcie funkcję kontrfakty(r, niezmienne, max_zmian=2), która szuka najmniejszych zbiorów zmian
# (średnia rośnie co 0,05, przełączenie ects_w_terminie, hackathon, kolo, urlop) odwracających odmowę.
# Zastosujcie dlaczego() i kontrfakty() do studenta o indeksie IDX = 76, najpierw z urlopem jako cechą niezmienialną, potem bez tego ograniczenia.

In [ ]:
# Wybierzcie osobę bez urlopu, z odmową, faktycznie wyróżniającą się i ze średnią co najmniej 4,3. Powtórzcie analizę.

### 4b. Ta sama decyzja w sieci bayesowskiej

Struktura sieci jest dana: `Urlop -> Srednia`, `Urlop -> ECTS`, a `Srednia`, `ECTS`, `Kolo` i `Hackathon` są rodzicami `Wyroznienie`. Średnia jest zdyskretyzowana progami regulaminu. Wnioskowanie liczone jest przez jawne wymnożenie tablic CPT do rozkładu łącznego (sześć zmiennych binarnych lub trójwartościowych, więc to tanie i nie wymaga dodatkowych modułów).

In [ ]:
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.parameter_estimator import DiscreteBayesianEstimator

tak_nie = {True: "tak", False: "nie"}
bn_df = pd.DataFrame({
    "Urlop": pop.urlop.map(tak_nie),
    "Srednia": pd.cut(pop.srednia, [0, 4.2, 4.6, 5.01], right=False, labels=["niska", "dobra", "wysoka"]).astype(str),
    "ECTS": pop.ects_w_terminie.map(tak_nie),
    "Kolo": pop.kolo.map(tak_nie),
    "Hackathon": pop.hackathon.map(tak_nie),
    "Wyroznienie": pop.wyroznienie.map(tak_nie),
})
KRAWEDZIE = [("Urlop", "Srednia"), ("Urlop", "ECTS"), ("Srednia", "Wyroznienie"),
             ("ECTS", "Wyroznienie"), ("Kolo", "Wyroznienie"), ("Hackathon", "Wyroznienie")]

In [ ]:
# Nauczcie sieć (DiscreteBayesianNetwork, estymator DiscreteBayesianEstimator z prior_type="BDeu", equivalent_sample_size=10).
# Napiszcie p_wyr(dowody): rozkład łączny z iloczynu czynników CPT, redukcja do dowodów, marginalizacja, normalizacja.
# Dla studenta IDX policzcie posterior przy wszystkich dowodach i po usunięciu każdego dowodu z osobna.
# Wkład dowodu = różnica log-szans z dowodem i bez niego.

In [ ]:
# Policzcie wartości Shapleya wkładów dowodów do log-szans (średnia po wszystkich 120 kolejnościach dodawania dowodów).
# Porównajcie ich sumę z różnicą logit(posterior) - logit(prior) oraz z sumą wkładów z poprzedniej komórki.

### Wnioski do zadania 4

(uzupełnij)

## Zadanie 5: audyt uczciwości regulaminu

Atrybut chroniony: `plec`. Regulamin go nie używa.

In [ ]:
# Napiszcie wektorową wersję regulaminu z parametrami (progi, blokada urlopu, rodzaj wymaganej aktywności).
# Napiszcie audyt(dec): liczba przyznanych, stopy selekcji K i M, demographic parity (różnica),
# disparate impact (iloraz K/M), TPR w grupach i różnica equal opportunity, precyzja i czułość względem "wyroznienie".
# Czy spełniona jest reguła 80%?

In [ ]:
# Szukajcie zmiennych zastępczych (proxy): porównajcie średnie cech w grupach płci
# i policzcie AUC regresji logistycznej przewidującej płeć z pojedynczych cech oraz ze wszystkich (5-krotna walidacja).
# Skąd bierze się zależność hackathonu od płci? (tabela hackathon wg kierunku i płci)

In [ ]:
# Zaproponujcie poprawki regulaminu: usunięcie R1, zastąpienie warunku "hackathon" przez "hackathon lub koło", obie zmiany.
# Potem dobierzcie progi (siatka co 0,05) tak, żeby liczba stypendiów różniła się od regulaminowej najwyżej o 10%,
# a precyzja była możliwie wysoka. Zestawcie audyty wszystkich wariantów w jednej tabeli.

In [ ]:
# Wykres słupkowy: stopy przyznań i TPR w grupach K i M dla wszystkich wariantów.

### Wnioski do zadania 5

(uzupełnij)

## Zadanie 6: RODO i AI Act

Wypełnijcie tabelę dla systemów z zadań 1 do 5 (graf uczelni z publikacją, system stypendialny). W kolumnie „przepis” podawajcie tylko te jednostki redakcyjne, które sami odnaleźliście w tekście aktu na EUR-Lex.

| pytanie | odpowiedź | przepis |
|---|---|---|
| Czy pseudonimizowany graf z zadania 1 to dane osobowe? | | |
| Czy informacja o orzeczeniu to dane szczególnej kategorii? | | |
| Czy system stypendialny dokonuje profilowania? | | |
| Czy decyzja o stypendium jest „opartą wyłącznie na zautomatyzowanym przetwarzaniu”? Jakie prawa ma student? | | |
| Czy regulamin regułowy jest systemem AI w rozumieniu AI Act? | | |
| Do jakiej kategorii ryzyka należy system wspomagający decyzje w edukacji? | | |
| Jakie obowiązki z AI Act odpowiadają zadaniom 2 do 5? | | |

## Zadanie 7: Pytania kontrolne

Odpowiedzi na pytania kontrolne z instrukcji, po jednym do trzech zdań.

1. 
2. 
3. 
4. 
5. 
6. 
7. 